# Preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# create target directory
!mkdir -p /content/imgs

# unzip to /content/imgs
!unzip -q "/content/drive/MyDrive/imgs.zip" -d /content/imgs

# check file structure
!ls /content/imgs/train
!ls /content/imgs/test

# set paths
train_dir = "/content/imgs/train"  # see folders of c0, c1, c2, etc.
test_dir = "/content/imgs/test"  # image files

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms, models, datasets
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import os
import cv2
import random
import glob
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

In [ ]:
train_dir = "/content/imgs/train"
test_dir = "/content/imgs/test"

In [ ]:
# data augmentation
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# load the training set (automatically gets the categories from the folder structure)
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
class_names = train_dataset.classes  # automatically obtain the class names such as c0, c1, c2...
num_classes = len(class_names)

In [ ]:
class TestDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.transform = transform
        self.image_paths = [
            os.path.join(folder_path, f)
            for f in os.listdir(folder_path)
            if f.lower().endswith(('.jpg', '.png', '.jpeg'))
        ]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)  # preprocess images
        return image, 0  # return image and a pseudo label 0 (remember test_dataset have no labels)

In [ ]:
# create a data loader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)  # shuffle=True for generalization
test_dataset = TestDataset(test_dir, transform=test_transform)  # test_dataset has no labels

# test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)  # shuffle=False for consistency
# test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [ ]:
# define the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = models.resnet18(pretrained=True)
# model.fc = nn.Linear(model.fc.in_features, num_classes)
model = models.vgg16(pretrained=True)  # pretrained VGG16 model
model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)  # classifier[6] as the last layer of VGG16 - fully connected layer
model = model.to(device)

In [ ]:
# train function
def train_model(model, train_loader, criterion, optimizer, epochs=5):  # train_loader: DataLoader
    model.train()  # train mode
    for epoch in range(epochs):  # initialization
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()  # avoid accumulation
            outputs = model(images)  # forward pass with outputs
            loss = criterion(outputs, labels)  # loss calculation
            loss.backward()  # backward pass
            optimizer.step()  # weight updates

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Acc: {100*correct/total:.2f}%")

In [ ]:
# evaluation function (validation only on the training set)
def evaluate(model, dataloader):
    model.eval()  # evaluation mode
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)  # forward pass
            _, preds = torch.max(outputs, 1)  # return index

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.show()

# Modeling

In [ ]:
# train the model
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)  # momentum for faster convergence

In [ ]:
print("begin training...")
train_model(model, train_loader, criterion, optimizer, epochs=5)

In [ ]:
# evaluate the model (split validation set on training set)
from torch.utils.data import random_split
import torch

#seed = 42
#generator = torch.Generator().manual_seed(seed)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
#train_subset, val_subset = random_split(train_dataset, [train_size, val_size], generator=generator)
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

In [ ]:
val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)
print("\nValidation set evaluation:")
evaluate(model, val_loader)

In [ ]:
# set random seeds
seed = 42
g = torch.Generator()
g.manual_seed(seed)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, generator=g)

In [ ]:
 # visualize predictions on the test set (without true labels)
def visualize_test_predictions(model, test_loader, class_names, num_samples=9):  # output 9 plots
    model.eval()  # evaluation mode
    images_shown = 0

    plt.figure(figsize=(12, 12))
    with torch.no_grad():  # disable gradient calculation
        for images, _ in test_loader:
            images = images.to(device)
            outputs = model(images)  # get predictions
            _, preds = torch.max(outputs, 1)  # get predicted class index

            for i in range(images.size(0)):
                if images_shown >= num_samples:  # stop if >=9
                    break

                img = images[i].cpu().numpy().transpose((1, 2, 0))  # convert image tensor to numpy and transpose to HWC
                img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])  # unnormalize
                img = np.clip(img, 0, 1)  # valid range [0, 1] for pixel values

                plt.subplot(3, 3, images_shown+1)
                plt.imshow(img)
                plt.title(f"Pred: {class_names[preds[i]]}")
                plt.axis('off')
                images_shown += 1

            if images_shown >= num_samples:
                break

    plt.tight_layout()
    plt.show()

print("\nTest set prediction example:")

visualize_test_predictions(model, test_loader, class_names)

# Grad-CAM

In [ ]:
# Grad-CAM
def apply_gradcam(model, img_tensor, target_layer):
    features = {}  # store forward pass outputs
    grads = {}  # store backward pass outputs

    def forward_hook(module, input, output):
        features['value'] = output.detach()

    def backward_hook(module, grad_input, grad_output):
        grads['value'] = grad_output[0].detach()

    # register hook function to the target level
    hook_forward = target_layer.register_forward_hook(forward_hook)
    hook_backward = target_layer.register_backward_hook(backward_hook)

    # forward pass
    model.eval()
    output = model(img_tensor.unsqueeze(0).to(device))
    pred_class = output.argmax(dim=1).item()

    # backward pass
    model.zero_grad()
    one_hot = torch.zeros_like(output)  # one-hot
    one_hot[0][pred_class] = 1
    output.backward(gradient=one_hot)

    # remove hook
    hook_forward.remove()
    hook_backward.remove()

    # calculate Grad-CAM
    activations = features['value']
    gradients = grads['value']
    pooled_gradients = torch.mean(gradients, dim=[0, 2, 3])

    # average gradients
    for i in range(activations.shape[1]):
        activations[:, i, :, :] *= pooled_gradients[i]

    # weighted
    heatmap = torch.mean(activations, dim=1).squeeze()  # squeeze into one image
    heatmap = F.relu(heatmap)
    heatmap /= torch.max(heatmap)

    # return heatmap and classification
    return heatmap.cpu().numpy(), pred_class


# image preprocessing
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


# Grad-CAM target layer
# target_layer = model.layer4[-1].conv2
target_layer = model.features[-1]  # the last convolutional layer


# load image paths
test_images = glob.glob(os.path.join(test_dir, '**', '*.*'), recursive=True)
test_images = [f for f in test_images if f.lower().endswith(('.jpg', '.jpeg', '.png'))]


random.seed(42)
selected_images = random.sample(test_images, min(5, len(test_images)))

for img_path in selected_images:
    img = Image.open(img_path).convert('RGB')
    img_tensor = preprocess(img).to(device)

    heatmap, pred_class = apply_gradcam(model, img_tensor, target_layer)

    plt.figure(figsize=(12, 5))

    # original image
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f"Original\nPredicted class: {pred_class}")
    plt.axis('off')

    # resize and apply the heatmap
    heatmap = cv2.resize(heatmap, img.size)
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    # superimpose heatmap on original image
    superimposed = cv2.addWeighted(np.array(img), 0.6, heatmap, 0.4, 0)

    plt.subplot(1, 2, 2)
    plt.imshow(superimposed)
    plt.title("Grad-CAM")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# Bias Check



In [ ]:
import zipfile
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

In [ ]:
# manual image selection to form new datasets
zip_path = "/content/drive/MyDrive/newimgs10.zip"
extract_path = "/content/newimgs10"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
# repeat procedures
# extract_path = "/content/newimgs/newimgs"
extract_path = "/content/newimgs10/newimgs10"

for root, dirs, files in os.walk(extract_path):
    image_files = glob.glob(os.path.join(root, '*.*'))

    image_files = [f for f in image_files if f.lower().endswith(('png', 'jpg', 'jpeg', 'tiff', 'bmp'))]

    print(f"{root}: {len(image_files)} images")

In [ ]:
# data augmentation
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
# Grad-CAM
def apply_gradcam(model, img_tensor, target_layer):
    features, grads = {}, {}  # store forward and backward pass outputs

    def forward_hook(m, i, o): features['value'] = o.detach()
    def backward_hook(m, gi, go): grads['value'] = go[0].detach()

    # register hook function to the target level
    h1 = target_layer.register_forward_hook(forward_hook)
    h2 = target_layer.register_backward_hook(backward_hook)

    # forward pass + backward pass
    output = model(img_tensor.unsqueeze(0).to(device))
    pred_class = output.argmax(dim=1).item()
    one_hot = torch.zeros_like(output)
    one_hot[0][pred_class] = 1
    model.zero_grad()
    output.backward(gradient=one_hot)

    # remove hook
    h1.remove(); h2.remove()

    act, grad = features['value'], grads['value']  # calculate Grad-CAM
    pooled_grad = torch.mean(grad, dim=[0, 2, 3])
    for i in range(act.shape[1]):  # average gradients
        act[:, i, :, :] *= pooled_grad[i]

    heatmap = torch.mean(act, dim=1).squeeze().cpu()  # squeeze into one image
    heatmap = torch.clamp(heatmap, min=0)
    heatmap /= torch.max(heatmap)
    return heatmap.numpy(), pred_class  # return heatmap and classification

In [ ]:
results = defaultdict(lambda: {'true': [], 'pred': [], 'paths': []})
target_layer = model.features[-1]  # the last convolutional layer

categories = sorted(os.listdir(extract_path))  # c0 - c9
for cat in categories:
    for light_type in ['white', 'non-white']:  # iterate over skin type / race
        folder = os.path.join(extract_path, cat, light_type)  # path to images
        image_paths = glob.glob(os.path.join(folder, '*.*'))

        for img_path in tqdm(image_paths, desc=f"{cat}/{light_type}"):
            try:
                img = Image.open(img_path).convert('RGB')  # load and convert image to RGB
                img_tensor = preprocess(img).to(device)  # preprocessing + move to GPU
                with torch.no_grad():
                    output = model(img_tensor.unsqueeze(0))
                    pred = output.argmax(dim=1).item()  # get predicted class index

                true = int(cat[1])  # extract true label from category name [c0 -> 0]
                results[f'{cat}_{light_type}']['true'].append(true)
                results[f'{cat}_{light_type}']['pred'].append(pred)
                results[f'{cat}_{light_type}']['paths'].append(img_path)

            except Exception as e:
                print(f"Error loading {img_path}: {e}")

In [ ]:
# classification report
for key in results:
    y_true = results[key]['true']
    y_pred = results[key]['pred']
    print(f"==== Results for {key} ====")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='macro'):.4f}")
    print(f"Recall: {recall_score(y_true, y_pred, average='macro'):.4f}")
    print(f"F1-score: {f1_score(y_true, y_pred, average='macro'):.4f}")
    print()

In [ ]:
# assign key lists for viz
target_keys = ['c3_non-white']

for key in results:
    if key in target_keys:
        print(f"Visualizing: {key}")
        for img_path in results[key]['paths']:
            img = Image.open(img_path).convert('RGB')
            img_tensor = preprocess(img).to(device)
            heatmap, pred_class = apply_gradcam(model, img_tensor, target_layer)

            # heatmap
            heatmap = cv2.resize(heatmap, img.size)
            heatmap = np.uint8(255 * heatmap)
            heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

            # add heatmap to original
            superimposed = cv2.addWeighted(np.array(img), 0.6, heatmap, 0.4, 0)

            # viz
            plt.figure(figsize=(8, 8))
            plt.imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))  # BGR -> RGB
            plt.title(f"Grad-CAM: {key}")
            plt.axis('off')
            plt.show()

In [ ]:
# assign key lists for viz
target_keys = ['c4_non-white']

for key in results:
    if key in target_keys:
        print(f"Visualizing: {key}")
        for img_path in results[key]['paths']:
            img = Image.open(img_path).convert('RGB')
            img_tensor = preprocess(img).to(device)
            heatmap, pred_class = apply_gradcam(model, img_tensor, target_layer)

            # heatmap
            heatmap = cv2.resize(heatmap, img.size)
            heatmap = np.uint8(255 * heatmap)
            heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

            # add heatmap to original
            superimposed = cv2.addWeighted(np.array(img), 0.6, heatmap, 0.4, 0)

            # viz
            plt.figure(figsize=(8, 8))
            plt.imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))  # BGR -> RGB
            plt.title(f"Grad-CAM: {key}")
            plt.axis('off')
            plt.show()

In [ ]:
# assign key lists for viz
target_keys = ['c5_non-white']

for key in results:
    if key in target_keys:
        print(f"Visualizing: {key}")
        for img_path in results[key]['paths']:
            img = Image.open(img_path).convert('RGB')
            img_tensor = preprocess(img).to(device)
            heatmap, pred_class = apply_gradcam(model, img_tensor, target_layer)

            # heatmap
            heatmap = cv2.resize(heatmap, img.size)
            heatmap = np.uint8(255 * heatmap)
            heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

            # add heatmap to original
            superimposed = cv2.addWeighted(np.array(img), 0.6, heatmap, 0.4, 0)

            # viz
            plt.figure(figsize=(8, 8))
            plt.imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))  # BGR -> RGB
            plt.title(f"Grad-CAM: {key}")
            plt.axis('off')
            plt.show()

In [ ]:
# assign key lists for viz
target_keys = ['c6_non-white']

for key in results:
    if key in target_keys:
        print(f"Visualizing: {key}")
        for img_path in results[key]['paths']:
            img = Image.open(img_path).convert('RGB')
            img_tensor = preprocess(img).to(device)
            heatmap, pred_class = apply_gradcam(model, img_tensor, target_layer)

            # heatmap
            heatmap = cv2.resize(heatmap, img.size)
            heatmap = np.uint8(255 * heatmap)
            heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

            # add heatmap to original
            superimposed = cv2.addWeighted(np.array(img), 0.6, heatmap, 0.4, 0)

            # viz
            plt.figure(figsize=(8, 8))
            plt.imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))  # BGR -> RGB
            plt.title(f"Grad-CAM: {key}")
            plt.axis('off')
            plt.show()

In [ ]:
# assign key lists for viz
target_keys = ['c9_non-white']

for key in results:
    if key in target_keys:
        print(f"Visualizing: {key}")
        for img_path in results[key]['paths']:
            img = Image.open(img_path).convert('RGB')
            img_tensor = preprocess(img).to(device)
            heatmap, pred_class = apply_gradcam(model, img_tensor, target_layer)

            # heatmap
            heatmap = cv2.resize(heatmap, img.size)
            heatmap = np.uint8(255 * heatmap)
            heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

            # add heatmap to original
            superimposed = cv2.addWeighted(np.array(img), 0.6, heatmap, 0.4, 0)

            # viz
            plt.figure(figsize=(8, 8))
            plt.imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))  # BGR -> RGB
            plt.title(f"Grad-CAM: {key}")
            plt.axis('off')
            plt.show()

In [ ]:
# Finding: non-white images with low accuracy -> model pays attention to the surroundings (c4, c5, c9) / clothes (c3, c6, c9)